In [61]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [62]:
import numpy as np
import json
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.figsize": (6, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.xmargin": 0,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.size": 11,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "legend.frameon": True,
})

# mpl.rcParams.update({
#     "figure.figsize": (6, 4),
#     "axes.spines.top": False,
#     "axes.spines.right": False,
#     "axes.facecolor": "#fafafa",
#     "axes.grid": True,
#     "grid.color": "#dddddd",
#     "grid.linestyle": "--",
#     "axes.prop_cycle": mpl.cycler(color=["#4C72B0", "#55A868", "#C44E52",
#                                          "#8172B2", "#CCB974", "#64B5CD"]),
#     "font.size": 11,
#     "axes.labelsize": 12,
#     "axes.titlesize": 14,
#     "legend.frameon": True,
#     "legend.facecolor": "white",
#     "legend.edgecolor": "#dddddd",
#     "lines.linewidth": 2,
# })

from plt_utils import plot, savefig

import os

# task_name, data_name, default_window, default_token_dim = "assoc-recall-mk", "data_100_0_8", 20, 8
# task_name, data_name, default_window, default_token_dim = "assoc-recall-mk", "data_100_0_8", 20, 16
task_name, data_name, default_window, default_token_dim = "assoc-recall-mk", "data_100_0_8", 100, 16
# task_name, data_name, default_window, default_token_dim = "decode-recall", "data_100_2_32", 100, 24
# task_name, data_name, default_window, default_token_dim = "decode-recall-last", "data_100_2_32", 100, 24
# task_name, data_name, default_window, default_token_dim = "var-copy", "data_100_5_26", 20, 8 # 20, 24

In [63]:
dashed_task_name = '-'.join(task_name.split('_'))

all_losses = {}
all_accs = {}
all_train_accs = {}
all_params = {}

In [64]:
for run_name in os.listdir("results/" + task_name + "/" + data_name):
    if len(run_name.split("_")) != 7: continue

    # if int(run_name.split("_")[4][1:]) > 50: # Filter for large dimension runs
        # continue

    for run in os.listdir("results/" + task_name + "/" + data_name + "/" + run_name):
        if run.startswith("."):
            continue

        # print("results/" + task_name + "/" + data_name + "/" + run_name + "/" + run)
        # data = torch.load("results/" + task_name + "/" + data_name + "/" + run_name + "/" + run, weights_only=False)
        with open("results/" + task_name + "/" + data_name + "/" + run_name + "/" + run) as infile:
            data = json.load(infile)
        loss = data["final_loss"]
        accs = data["train_accs"]
        print(accs)
        acc = data["final_acc"]
        # args = data["args"]
        params = data["params"]

        if run_name not in all_losses.keys():
            all_losses[run_name] = [loss]
            all_accs[run_name] = [acc[-1]]
            all_params[run_name] = params
            all_train_accs[run_name] = [accs]
        else:
            all_losses[run_name].append(loss)
            all_accs[run_name].append(acc[-1])
            all_train_accs[run_name].append(accs)

for run_name in all_losses.keys():
    all_losses[run_name] = np.array(all_losses[run_name])
    all_accs[run_name] = np.array(all_accs[run_name])
    all_train_accs[run_name] = np.array(all_train_accs[run_name])

[0.5509542226791382, 0.5378150343894958, 0.5883121490478516, 0.5544863939285278]
[0.40055233240127563, 0.36914095282554626, 0.3605211079120636, 0.39138180017471313]
[0.5295699834823608, 0.5220794081687927, 0.5120234489440918, 0.5267735719680786]
[0.5199825763702393, 0.5360926389694214, 0.5141145586967468, 0.5122722387313843]
[0.5382618308067322, 0.5112200975418091, 0.5536389350891113, 0.523198127746582]
[0.5920188426971436, 0.6212859153747559, 0.5614680051803589, 0.6486009955406189]
[0.4751615524291992, 0.5029705166816711, 0.5185073614120483, 0.46285176277160645]
[0.41640082001686096, 0.42883649468421936, 0.400158166885376, 0.4669591188430786]
[0.5064279437065125, 0.5293078422546387, 0.5132164359092712, 0.5315434336662292]
[0.48323923349380493, 0.4944406747817993, 0.46400779485702515, 0.5443179607391357]
[0.6436766982078552, 0.631529688835144, 0.6071503162384033, 0.5696085095405579]
[0.4664038419723511, 0.5326696634292603, 0.466636598110199, 0.44617971777915955]
[0.5133059024810791, 0.

In [72]:
state_dim = 1
num_heads = 1
dim = 24 # default_token_dim
window = 100 # Default window
for k, v in all_accs.items():
    split = k.split("_")
    if split[2] not in ["SSM-SSM", "SSM-TF", "TF-SSM", "TF-TF"]:
        continue    
    if split[-4] != "w%d" % default_window:
        continue
    if split[-3] != "d%d" % dim:
        continue
    if split[-2] != "nh%d" % num_heads:
        continue
    if split[-1] != "sd%d" % state_dim:
        continue
    print(k, all_params[k], np.mean(v))

run_assoc-recall-mk_TF-SSM_w100_d24_nh1_sd1 11496 0.5239278945055875
run_assoc-recall-mk_SSM-SSM_w100_d24_nh1_sd1 12936 0.5170181285251271
run_assoc-recall-mk_SSM-TF_w100_d24_nh1_sd1 11496 0.9891958724368702
run_assoc-recall-mk_TF-TF_w100_d24_nh1_sd1 10056 0.6680852215398442


In [66]:
np.unique(np.array([k.split("_")[-3] for k in all_accs.keys()]))

array(['d12', 'd16', 'd20', 'd24', 'd4', 'd8'], dtype='<U3')